In [1]:
# ── 셀 1 │ 설정 ──
import os, sys, json, time, random, urllib.request, urllib.parse
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import add_self_loops, softmax
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

NEW  = r'C:\Users\iisl\Desktop\0_Molecualar_Informatics\2026_final\2026_new' + '\\'
CLIN = NEW + r'data\processed\clinical\clinical_pool_endpoints.csv'
CMP  = NEW + r'data\processed\crediblemeds\crediblemeds_pool.csv'
IKC  = NEW + r'data\processed\crediblemeds\name2key_cache.json'
SMIC = NEW + r'data\processed\crediblemeds\crediblemeds_smiles.csv'
PRE  = NEW + r'03_dedup\pretrain_drop.pt'
RES  = NEW + r'results\arm_compare\\'
os.makedirs(RES, exist_ok=True)
for p in [CLIN, CMP, PRE]:
    assert os.path.exists(p), p

sys.path.insert(0, NEW + '01_pre-train')
import featurizer as FZ

DEV   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = (42, 43, 44)
HID, HEADS, DROP = 32, 8, 0.3           # SI 기술값. 사전학습과 같아야 한다
FT_LR_BB, FT_LR_CLF, FT_EPOCHS, FT_PAT, FT_BS = 1e-4, 5e-3, 60, 12, 32
THR   = 0.50

ARMS = {'Transfer learned': PRE,
        'Clinical only'   : None}      # None = 무작위 초기화

print(DEV, '| atom', FZ.num_atom_features())

C:\Users\iisl\anaconda3\envs\esu_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda | atom 39


In [2]:
# ── 셀 2 │ GAT_SI ──
class GH(nn.Module):
    def __init__(s, ic, oc, d=DROP):
        super().__init__()
        s.W = nn.Parameter(torch.empty(ic, oc))
        s.a = nn.Parameter(torch.empty(2 * oc, 1))
        nn.init.xavier_uniform_(s.W); nn.init.xavier_uniform_(s.a)
        s.lr = nn.LeakyReLU(0.2); s.d = d
    def forward(s, x, ei):
        ei, _ = add_self_loops(ei, num_nodes=x.size(0))
        h = x @ s.W
        row, col = ei
        e = s.lr((torch.cat([h[row], h[col]], -1) @ s.a).squeeze(-1))
        e = F.dropout(e, s.d, training=s.training)
        al = softmax(e, row)
        o = torch.zeros_like(h)
        o.index_add_(0, row, h[col] * al.unsqueeze(-1))
        return F.elu(o)

class GL(nn.Module):
    def __init__(s, ic, oc, h, d=DROP):
        super().__init__()
        s.h, s.d = h, d
        s.att = nn.ModuleList([GH(ic, oc, d) for _ in range(h)])
    def forward(s, x, ei):
        z = (torch.mean(torch.stack([a(x, ei) for a in s.att], 0), 0)
             if s.h == 1 else torch.cat([a(x, ei) for a in s.att], -1))
        return F.dropout(z, s.d, training=s.training)

class AttentionPool(nn.Module):
    def __init__(s, ch, d=DROP):
        super().__init__()
        s.gate = nn.Sequential(nn.Linear(ch, ch), nn.ReLU(),
                               nn.Dropout(d), nn.Linear(ch, 1))
    def forward(s, x, batch):
        al = softmax(s.gate(x).squeeze(-1), batch)
        out = torch.zeros(int(batch.max()) + 1, x.size(1),
                          device=x.device, dtype=x.dtype)
        out.index_add_(0, batch, x * al.unsqueeze(-1))
        return out

class GAT_SI(nn.Module):
    def __init__(s, ic, hc=HID, h=HEADS, d=DROP):
        super().__init__()
        s.l1 = GL(ic, hc, h, d)
        s.l2 = GL(hc * h, hc, h, d)
        s.l3 = GL(hc * h, hc, 1, d)
        s.pool = AttentionPool(hc, d)
        s.clf = nn.Sequential(nn.Linear(hc, 64), nn.ReLU(),
                              nn.Dropout(d), nn.Linear(64, 1))
    def forward(s, x, ei, b):
        x = F.elu(s.l1(x, ei)); x = F.elu(s.l2(x, ei)); x = s.l3(x, ei)
        return s.clf(s.pool(x, b)).squeeze(-1)

print('params', sum(p.numel() for p in GAT_SI(FZ.num_atom_features()).parameters()))

params 88066


In [3]:
# ── 셀 3 │ CredibleMeds SMILES ──
cm = pd.read_csv(CMP, encoding='utf-8-sig')
cm = cm.loc[:, ~cm.columns.duplicated()].drop_duplicates('key14')
print('CredibleMeds', len(cm), '| tier', dict(cm['tier'].value_counts()))

BASE  = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound'
PROPS = 'SMILES,ConnectivitySMILES,IsomericSMILES,CanonicalSMILES'
KEYS  = ['IsomericSMILES', 'SMILES', 'CanonicalSMILES', 'ConnectivitySMILES']

def _get(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'python-urllib'})
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())

def fetch_smiles(ik=None, name=None):
    urls = []
    if ik:   urls.append(f'{BASE}/inchikey/{ik}/property/{PROPS}/JSON')
    if name: urls.append(f'{BASE}/name/{urllib.parse.quote(str(name))}'
                         f'/property/{PROPS}/JSON')
    for url in urls:
        try:
            p = _get(url)['PropertyTable']['Properties'][0]
        except Exception:
            continue
        for k in KEYS:
            if p.get(k):
                return p[k]
    return None

if os.path.exists(SMIC):
    smi_df = pd.read_csv(SMIC, encoding='utf-8-sig')
    print('SMILES 캐시 로드', len(smi_df))
else:
    n2k = json.load(open(IKC, encoding='utf-8')) if os.path.exists(IKC) else {}
    ik_by_name = {k: v.get('inchikey') for k, v in n2k.items()}
    rec = []
    for i, (_, r) in enumerate(cm.iterrows()):
        ik = r.get('pubchem_ik') or ik_by_name.get(str(r.get('name_primary')))
        rec.append({'key14': r['key14'], 'name_primary': r.get('name_primary'),
                    'smiles': fetch_smiles(ik, r.get('name_primary'))})
        if (i + 1) % 25 == 0:
            print(f'{i+1}/{len(cm)}  확보 {sum(1 for x in rec if x["smiles"])}',
                  flush=True)
        time.sleep(0.25)
    smi_df = pd.DataFrame(rec)
    smi_df.to_csv(SMIC, index=False, encoding='utf-8-sig')

ok = smi_df['smiles'].notna().sum()
print(f'SMILES 확보 {ok} / {len(smi_df)}')
assert ok > 0, 'PubChem property 이름을 확인하세요'

CredibleMeds 270 | tier {'Possible': 152, 'Known': 67, 'Conditional': 51}
SMILES 캐시 로드 270
SMILES 확보 270 / 270


In [5]:
# ── 셀 4 │ 그래프 ──
def _prep(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None
    try: Chem.SanitizeMol(m)
    except Exception: return None
    try: Chem.AssignStereochemistry(m, cleanIt=True, force=True)
    except Exception: pass
    return m

def make_graph(smi, y=None):
    mol = _prep(smi)
    if mol is None: return None
    try: x = np.stack([FZ.atom_features(a) for a in mol.GetAtoms()])
    except Exception: return None
    src, dst, ea = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = FZ.bond_features(b)
        src += [i, j]; dst += [j, i]; ea += [bf, bf]
    if not src: return None
    g = Data(x=torch.tensor(np.asarray(x, np.float32)),
             edge_index=torch.tensor([src, dst], dtype=torch.long),
             edge_attr=torch.tensor(np.asarray(ea, np.float32)))
    g.y = torch.tensor([float(y if y is not None else 1.0)], dtype=torch.float)
    return g

def scaffold_of(smi):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smiles=str(smi),
                                                   includeChirality=False)
    except Exception:
        return ''

# ── 임상 ── CSV 행 순서를 유지해야 마스크가 맞는다
clin = pd.read_csv(CLIN, encoding='utf-8-sig')
for c in ['listed', 'cm_qt', 'cm_tdp']:
    if c in clin.columns:
        clin[c] = clin[c].astype(str).str.lower().isin(['true', '1'])

# scaffold 보강: 비고리 분자는 Murcko scaffold 가 빈 문자열 → NaN 으로 읽힌다
sc = (clin['scaffold'] if 'scaffold' in clin.columns
      else pd.Series([None] * len(clin), index=clin.index))
sc = sc.astype(str).str.strip().replace({'nan': '', 'None': '', '<NA>': ''})
need = sc.str.len() == 0
if need.any():
    sc.loc[need] = clin.loc[need, 'canon_smiles'].map(scaffold_of)
    sc = sc.astype(str).str.strip()
    still = sc.str.len() == 0
    sc.loc[still] = clin.loc[still, 'canon_smiles']
    print(f'scaffold 보강 {int(need.sum())}종 (그중 비고리 {int(still.sum())}종)')
clin['scaffold'] = sc.values
assert not clin['scaffold'].astype(str).str.strip().eq('').any()

Cg, keep = [], []
for i, r in clin.iterrows():
    g = make_graph(r['canon_smiles'], r['y'])
    if g is None: continue
    Cg.append(g); keep.append(i)
M = clin.loc[keep].reset_index(drop=True)
y = M['y'].astype(int).values
print('임상', len(Cg), dict(M['y'].value_counts()),
      '| scaffold', M['scaffold'].nunique())

# ── 외부 ── 임상 풀 밖 CredibleMeds
ext = (smi_df.dropna(subset=['smiles'])
             .merge(cm[['key14', 'tier', 'QT Increase', 'TdP Cases']],
                    on='key14', how='left'))
ext = ext[~ext['key14'].isin(set(M['key14']))].drop_duplicates('key14')

ext_g, rows = [], []
for _, r in ext.iterrows():
    g = make_graph(r['smiles'], 1.0)
    if g is None: continue
    ext_g.append(g)
    rows.append({'key14': r['key14'], 'name': r['name_primary'],
                 'tier': r['tier'], 'qt': r['QT Increase'],
                 'tdp': r['TdP Cases']})
E = pd.DataFrame(rows)
print('외부', len(ext_g), '/', len(ext))
print(E['tier'].value_counts())

scaffold 보강 86종 (그중 비고리 86종)
임상 1225 {1: 725, 0: 500} | scaffold 921
외부 124 / 127
tier
Possible       80
Known          26
Conditional    18
Name: count, dtype: int64


In [6]:
# ── 셀 5 │ metrics / fit / folds ──
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def metrics(yt, p, thr=0.5):
    yh = (np.asarray(p) >= thr).astype(int); yt = np.asarray(yt).astype(int)
    return dict(accuracy=accuracy_score(yt, yh),
                auroc=roc_auc_score(yt, p) if len(set(yt)) > 1 else np.nan,
                aucpr=average_precision_score(yt, p) if len(set(yt)) > 1 else np.nan,
                f1=f1_score(yt, yh, zero_division=0),
                precision=precision_score(yt, yh, zero_division=0),
                recall=recall_score(yt, yh, zero_division=0))

@torch.no_grad()
def predict(model, loader):
    model.eval(); ps, ys = [], []
    for b in loader:
        b = b.to(DEV)
        ps += torch.sigmoid(model(b.x, b.edge_index, b.batch)).cpu().tolist()
        ys += b.y.view(-1).cpu().tolist()
    return np.array(ys), np.array(ps)

def fit(model, tr_ds, va_ds, opt, epochs, patience, bsz):
    tr = DataLoader(tr_ds, batch_size=bsz, shuffle=True)
    vl = DataLoader(va_ds, batch_size=256)
    npos = sum(int(g.y.item()) for g in tr_ds); nneg = len(tr_ds) - npos
    pw = torch.tensor([max(nneg, 1) / max(npos, 1)], dtype=torch.float).to(DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, state, w = -1.0, None, 0
    for _ in range(epochs):
        model.train()
        for b in tr:
            b = b.to(DEV); opt.zero_grad()
            crit(model(b.x, b.edge_index, b.batch),
                 b.y.view(-1).float()).backward()
            opt.step()
        ys, ps = predict(model, vl)
        a = roc_auc_score(ys, ps) if len(set(ys.astype(int))) > 1 else 0.5
        if a > best:
            best, w = a, 0
            state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            w += 1
            if w >= patience: break
    if state:
        model.load_state_dict({k: v.to(DEV) for k, v in state.items()})
    return model, best

def _reset(mod):
    if hasattr(mod, 'reset_parameters'):
        mod.reset_parameters()

def scaffold_folds(scaf, seed, k=5):
    s = pd.Series(scaf).astype(str)
    assert not s.str.strip().eq('').any(), 'scaffold 빈값이 있습니다'
    sizes = s.value_counts()
    names = list(sizes.index); np.random.RandomState(seed).shuffle(names)
    bins, load = [[] for _ in range(k)], np.zeros(k)
    for g in names:
        b = int(np.argmin(load)); bins[b].append(g); load[b] += sizes[g]
    out = []
    covered = np.zeros(len(s), bool)
    for b in bins:
        te = np.where(s.isin(b))[0]
        covered[te] = True
        out.append((np.setdiff1d(np.arange(len(s)), te), te))
    assert covered.all(), f'{(~covered).sum()}종이 어느 fold 에도 없습니다'
    return out

In [7]:
# ── 셀 6 │ 실행 ──
IC = FZ.num_atom_features()

def run_seed(ckpt, seed):
    folds = scaffold_folds(M['scaffold'].values, seed)
    oof, ext_p = np.full(len(Cg), np.nan), []
    for fi, (tr, te) in enumerate(folds):
        set_seed(seed)
        m = GAT_SI(IC).to(DEV)
        if ckpt is not None:
            sd = torch.load(ckpt, map_location=DEV, weights_only=True)
            miss, unexp = m.load_state_dict(sd, strict=False)
            assert all(k.startswith('clf.') for k in miss), miss
            assert not unexp, unexp
            m.clf.apply(_reset)
        trn, val = train_test_split(tr, test_size=0.10,
                                    random_state=seed, stratify=y[tr])
        def ds(ix):
            out = []
            for i in ix:
                g = Cg[i].clone()
                g.y = torch.tensor([float(y[i])], dtype=torch.float)
                out.append(g)
            return out
        bb = [p for n, p in m.named_parameters() if not n.startswith('clf.')]
        opt = torch.optim.Adam(
            [{'params': bb,                 'lr': FT_LR_BB},
             {'params': m.clf.parameters(), 'lr': FT_LR_CLF}],
            weight_decay=5e-4)
        m, _ = fit(m, ds(trn), ds(val), opt, FT_EPOCHS, FT_PAT, FT_BS)
        _, p = predict(m, DataLoader(ds(te), batch_size=256))
        oof[te] = p
        _, pe = predict(m, DataLoader(ext_g, batch_size=256))
        ext_p.append(pe)                   # 5 fold 모델 모두 외부에 눈멀어 있다
    assert not np.isnan(oof).any()
    return oof, np.mean(ext_p, axis=0)

t0 = time.time()
OOF, EXT = {}, {}
for arm, ck in ARMS.items():
    for sd in SEEDS:
        OOF[(arm, sd)], EXT[(arm, sd)] = run_seed(ck, sd)
        print(f'{arm} seed {sd} 완료 ({time.time()-t0:.0f}s)', flush=True)

for arm in ARMS:
    tag = arm.split()[0].lower()
    np.save(RES + f'oof_{tag}.npy', np.vstack([OOF[(arm, s)] for s in SEEDS]))
    np.save(RES + f'ext_{tag}.npy', np.vstack([EXT[(arm, s)] for s in SEEDS]))

Transfer learned seed 42 완료 (209s)
Transfer learned seed 43 완료 (377s)
Transfer learned seed 44 완료 (530s)
Clinical only seed 42 완료 (650s)
Clinical only seed 43 완료 (892s)
Clinical only seed 44 완료 (1002s)


In [8]:
# ── 셀 7 │ 외부 엔드포인트 표 ──
MET   = ['accuracy','auroc','aucpr','f1','precision','recall']
LABEL = ['Accuracy','AUCROC','AUCPR','f1 score','precision','recall']

def _yes(s): return s.astype(str).str.strip().str.lower().eq('yes')
e_rec   = (_yes(E['qt']) | _yes(E['tdp'])).values
e_blank = (E['qt'].isna() & E['tdp'].isna()).values
e_no    = (~e_rec) & (~e_blank)
neg     = (y == 0)

SUBSETS = {
    'Overall'            : np.ones(len(E), bool),
    'QT/TdP recorded'    : e_rec,
    'No QT/TdP recorded' : e_no,
    'Known risk'         : (E['tier'] == 'Known').values,
    'Conditional risk'   : (E['tier'] == 'Conditional').values,
    'Possible risk'      : (E['tier'] == 'Possible').values,
}

rows = []
for arm in ARMS:
    for name, mask in SUBSETS.items():
        if mask.sum() < 5: continue
        per = []
        for sd in SEEDS:
            p  = np.concatenate([EXT[(arm, sd)][mask], OOF[(arm, sd)][neg]])
            yy = np.concatenate([np.ones(mask.sum(), int),
                                 np.zeros(neg.sum(), int)])
            per.append(metrics(yy, p, thr=THR))
        agg = {m: (np.mean([x[m] for x in per]),
                   np.std ([x[m] for x in per], ddof=1)) for m in MET}
        rows.append([arm, name, int(mask.sum()), int(neg.sum()),
                     round(mask.sum() / (mask.sum() + neg.sum()), 3)] +
                    [f'{agg[m][0]:.4f} ± {agg[m][1]:.4f}' for m in MET])

tab = pd.DataFrame(rows, columns=['Model','Endpoint','n_pos','n_neg',
                                  'Prevalence'] + LABEL)
tab.to_csv(RES + 'table_external_endpoints.csv', index=False,
           encoding='utf-8-sig')
print(tab.to_string(index=False))
print(f'\n{e_blank.sum()} external compounds with both QT and TdP fields blank '
      f'are excluded from the QT/TdP rows.')

           Model           Endpoint  n_pos  n_neg  Prevalence        Accuracy          AUCROC           AUCPR        f1 score       precision          recall
Transfer learned            Overall    124    500       0.199 0.6672 ± 0.0305 0.7461 ± 0.0125 0.3765 ± 0.0408 0.4620 ± 0.0112 0.3415 ± 0.0199 0.7177 ± 0.0370
Transfer learned    QT/TdP recorded     63    500       0.112 0.6578 ± 0.0355 0.7279 ± 0.0106 0.2212 ± 0.0269 0.3089 ± 0.0065 0.2001 ± 0.0090 0.6825 ± 0.0572
Transfer learned No QT/TdP recorded     19    500       0.037 0.6513 ± 0.0444 0.6742 ± 0.0120 0.0660 ± 0.0084 0.1061 ± 0.0167 0.0586 ± 0.0097 0.5614 ± 0.0804
Transfer learned         Known risk     26    500       0.049 0.6622 ± 0.0447 0.7726 ± 0.0169 0.1357 ± 0.0253 0.1927 ± 0.0208 0.1095 ± 0.0134 0.8077 ± 0.0000
Transfer learned   Conditional risk     18    500       0.035 0.6493 ± 0.0436 0.6521 ± 0.0076 0.0626 ± 0.0075 0.0903 ± 0.0171 0.0497 ± 0.0095 0.5000 ± 0.1111
Transfer learned      Possible risk     80    500   

In [1]:
import pandas as pd
from pathlib import Path

R = Path(r'C:\Users\iisl\Desktop\0_Molecualar_Informatics\2026_final\2026_new\results')

print('=' * 70)
print('A. 내부 분해  (results/external/table_clinical_endpoints.csv)')
print('   -> 지금 원고 Table 8 의 값')
print('=' * 70)
a = pd.read_csv(R / 'external' / 'table_clinical_endpoints.csv', encoding='utf-8-sig')
print(a[['Endpoint', 'n_pos', 'n_neg', 'AUCROC']].to_string(index=False))

print()
print('=' * 70)
print('B. 외부 검증  (results/arm_compare/table_external_endpoints.csv)')
print('   -> 3.5 본문의 0.7461 / 0.7287')
print('=' * 70)
b = pd.read_csv(R / 'arm_compare' / 'table_external_endpoints.csv', encoding='utf-8-sig')
print(b[['Model', 'Endpoint', 'n_pos', 'n_neg', 'Prevalence', 'AUCROC']].to_string(index=False))

print()
print('판정: A 의 양성 수는', sorted(a['n_pos'].unique()), '-> 임상 풀 1,225 종')
print('      B 의 양성 수는', sorted(b['n_pos'].unique()), '-> 임상 풀 밖 124 종 + 하위 등급')

A. 내부 분해  (results/external/table_clinical_endpoints.csv)
   -> 지금 원고 Table 8 의 값
          Endpoint  n_pos  n_neg        AUCROC
           Overall    725    500 0.6451±0.0052
        A (listed)    133    500 0.7967±0.0156
    B (not listed)    592    500 0.6110±0.0043
        C (A vs B)    133    592 0.7074±0.0060
A1 (QT/TdP record)     93    500 0.8129±0.0139
  A2 (explicit No)     30    500 0.7518±0.0160
           Overall    725    500 0.6328±0.0089
        A (listed)    133    500 0.7735±0.0120
    B (not listed)    592    500 0.6012±0.0090
        C (A vs B)    133    592 0.6838±0.0105
A1 (QT/TdP record)     93    500 0.7947±0.0121
  A2 (explicit No)     30    500 0.7029±0.0353

B. 외부 검증  (results/arm_compare/table_external_endpoints.csv)
   -> 3.5 본문의 0.7461 / 0.7287
           Model           Endpoint  n_pos  n_neg  Prevalence          AUCROC
Transfer learned            Overall    124    500       0.199 0.7461 ± 0.0125
Transfer learned    QT/TdP recorded     63    500       0.1